# BigNeuron Benchmark — Evaluation

| Metric | Description |
|--------|-------------|
| **ESA** | Bidirectional nearest-neighbor distance mean (↓) |
| **PDS** | Fraction of nodes exceeding 2-voxel threshold (↓) |
| **F1 / Precision / Recall** | Node-level, 2-voxel threshold |
| **av_fragmentation** | Fraction of gold edges with gap in auto (↓) |
| **av_contraction** | Euclidean / path-length per branch (↑ = straight, shortcut risk if >> gold) |
| **tips_error** | (auto_tips − gold_tips) / gold_tips (signed, 0에 가까울수록 좋음) |

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

ROOT        = Path('..').resolve()
GOLD_DIR    = ROOT / 'data' / 'gold_standard'
RESULTS_DIR = ROOT / 'results'
OUT_DIR     = ROOT / 'methods' / 'ours' / 'output'

METHODS       = ['ours', 'vaa3d']
METHOD_COLORS = {'ours': '#4c9be8', 'vaa3d': '#e87b4c'}

samples = sorted(
    set(p.stem for p in (RESULTS_DIR / 'ours').glob('*.swc'))
  & set(p.stem for p in GOLD_DIR.glob('*.swc'))
)
print(f'Samples: {samples}')

## Helper functions

In [ ]:
def read_pixelsize(stem):
    txt = (GOLD_DIR / f'{stem}.pixelsize.txt').read_text()
    m = re.search(r'Voxel size:\s*([\d.]+)x[\d.]+x([\d.]+)', txt, re.I)
    return float(m.group(1)), float(m.group(2))  # vxy, vz (µm)


def load_swc(path, scale_xy=1.0, scale_z=1.0):
    """SWC 로드 → nodes dict {id: {x,y,z,r,pid}} (µm 단위)"""
    nodes = {}
    for line in Path(path).read_text().splitlines():
        if line.startswith('#') or not line.strip():
            continue
        p = line.split()
        if len(p) < 7:
            continue
        nid = int(p[0])
        nodes[nid] = dict(
            x=float(p[2]) * scale_xy,
            y=float(p[3]) * scale_xy,
            z=float(p[4]) * scale_z,
            r=float(p[5]),
            pid=int(p[6]),
        )
    return nodes


def build_children(nodes):
    ch = {nid: [] for nid in nodes}
    for nid, n in nodes.items():
        if n['pid'] in ch:
            ch[n['pid']].append(nid)
    return ch


def pos_array(nodes):
    ids = sorted(nodes)
    arr = np.array([[nodes[i]['x'], nodes[i]['y'], nodes[i]['z']] for i in ids],
                   dtype=np.float32)
    return arr, ids


def count_tips(nodes, children):
    return sum(1 for nid in nodes
               if not children.get(nid) and nodes[nid]['pid'] != -1)

## Metric functions

In [ ]:
def compute_esa_pds_f1(auto_nodes, gold_nodes, threshold_um):
    """ESA / PDS / F1 / Precision / Recall (node-level, 2-voxel threshold)"""
    a_arr, _ = pos_array(auto_nodes)
    g_arr, _ = pos_array(gold_nodes)
    g_tree = cKDTree(g_arr)
    a_tree = cKDTree(a_arr)
    d_a2g, _ = g_tree.query(a_arr)   # 각 auto 노드 → 가장 가까운 gold
    d_g2a, _ = a_tree.query(g_arr)   # 각 gold 노드 → 가장 가까운 auto

    esa       = float((d_a2g.mean() + d_g2a.mean()) / 2)
    pds       = float((d_a2g > threshold_um).sum() + (d_g2a > threshold_um).sum()) \
                / (len(d_a2g) + len(d_g2a))
    precision = float((d_a2g <= threshold_um).mean())
    recall    = float((d_g2a <= threshold_um).mean())
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return dict(esa=esa, pds=pds, f1=f1, precision=precision, recall=recall)


def compute_fragmentation(gold_nodes, gold_children, auto_nodes, threshold_um,
                           step_um=1.0):
    """gold 각 edge를 step_um 간격으로 샘플링 → threshold 초과 구간 있는 edge 비율."""
    a_arr, _ = pos_array(auto_nodes)
    a_tree   = cKDTree(a_arr)
    n_frag = n_total = 0
    for pid, kids in gold_children.items():
        if pid not in gold_nodes:
            continue
        pn = gold_nodes[pid]
        for cid in kids:
            if cid not in gold_nodes:
                continue
            cn  = gold_nodes[cid]
            p0  = np.array([pn['x'], pn['y'], pn['z']])
            p1  = np.array([cn['x'], cn['y'], cn['z']])
            L   = float(np.linalg.norm(p1 - p0))
            if L < 1e-6:
                continue
            n_samp = max(2, int(L / step_um) + 1)
            samps  = p0 + np.linspace(0, 1, n_samp)[:, None] * (p1 - p0)
            d, _   = a_tree.query(samps)
            if (d > threshold_um).any():
                n_frag += 1
            n_total += 1
    return n_frag / n_total if n_total else float('nan')


def compute_contraction(nodes, children):
    """Branch-level contraction: mean(Euclidean / path_length) 각 branch segment.
    1.0 = 완벽히 직선. auto > gold 이면 shortcut 의심."""
    bifurc = {nid for nid, ch in children.items() if len(ch) >= 2}
    roots  = [nid for nid, n in nodes.items() if n['pid'] == -1]
    starts = set(roots) | bifurc | {c for nid in bifurc for c in children.get(nid, [])}

    vals = []
    for start in starts:
        if start not in nodes:
            continue
        cur, path_len = start, 0.0
        while True:
            kids = children.get(cur, [])
            if len(kids) != 1:
                break
            nxt = kids[0]
            n, p_ = nodes[nxt], nodes[cur]
            path_len += np.linalg.norm(
                [n['x']-p_['x'], n['y']-p_['y'], n['z']-p_['z']])
            cur = nxt
        if path_len < 1e-6:
            continue
        sn, se = nodes[start], nodes[cur]
        euclid = np.linalg.norm(
            [sn['x']-se['x'], sn['y']-se['y'], sn['z']-se['z']])
        vals.append(euclid / path_len)
    return float(np.mean(vals)) if vals else float('nan')


def compute_tips_error(auto_nodes, auto_ch, gold_nodes, gold_ch):
    """(auto_tips - gold_tips) / gold_tips (signed). 0에 가까울수록 좋음."""
    a_tips = count_tips(auto_nodes, auto_ch)
    g_tips = count_tips(gold_nodes, gold_ch)
    err    = (a_tips - g_tips) / g_tips if g_tips > 0 else float('nan')
    return err, a_tips, g_tips

## 평가 실행

In [ ]:
rows = []
for stem in samples:
    vxy, vz      = read_pixelsize(stem)
    threshold_um = 2.0       # 2 µm 절대값 (pyneval 기본값과 동일)

    gold    = load_swc(GOLD_DIR / f'{stem}.swc', scale_xy=vxy, scale_z=vz)
    gold_ch = build_children(gold)

    for method in METHODS:
        swc_path = RESULTS_DIR / method / f'{stem}.swc'
        if not swc_path.exists():
            continue

        if method == 'ours':
            auto = load_swc(swc_path)                            # 이미 µm
        else:
            auto = load_swc(swc_path, scale_xy=vxy, scale_z=vz) # pixel → µm
        auto_ch = build_children(auto)

        m       = compute_esa_pds_f1(auto, gold, threshold_um)
        frag    = compute_fragmentation(gold, gold_ch, auto, threshold_um)
        contr_a = compute_contraction(auto, auto_ch)
        contr_g = compute_contraction(gold, gold_ch)
        tip_err, a_tips, g_tips = compute_tips_error(auto, auto_ch, gold, gold_ch)

        rows.append(dict(
            sample=stem, method=method,
            esa=m['esa'], pds=m['pds'],
            f1=m['f1'], precision=m['precision'], recall=m['recall'],
            av_fragmentation=frag,
            av_contraction=contr_a,
            contraction_gold=contr_g,
            tips_auto=a_tips, tips_gold=g_tips, tips_error=tip_err,
            threshold_um=threshold_um,
        ))
        print(f'{method:8s}  {stem[:35]:35s}'
              f'  ESA={m["esa"]:5.2f}  PDS={m["pds"]:.3f}'
              f'  F1={m["f1"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}'
              f'  frag={frag:.3f}  contr={contr_a:.3f}(gold={contr_g:.3f})'
              f'  tips={a_tips}(gold={g_tips}) err={tip_err:+.2f}')

df = pd.DataFrame(rows)
df.to_csv('results_raw.csv', index=False)
df

## 요약 테이블

In [ ]:
MAIN_METRICS = ['esa', 'pds', 'f1', 'precision', 'recall']
AUX_METRICS  = ['av_fragmentation', 'av_contraction', 'tips_error']

summary = df.groupby('method')[MAIN_METRICS + AUX_METRICS].mean().round(4)
summary.style \
    .highlight_min(subset=['esa', 'pds', 'av_fragmentation'], color='lightgreen') \
    .highlight_max(subset=['f1', 'precision', 'recall', 'av_contraction'], color='lightgreen')

## 지표 비교 그래프

In [ ]:
plot_groups = [
    ('Primary',   ['esa', 'pds']),
    ('F1',        ['f1', 'precision', 'recall']),
    ('Auxiliary', ['av_fragmentation', 'av_contraction', 'tips_error']),
]
n_cols = max(len(g[1]) for g in plot_groups)
n_rows = len(plot_groups)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
palette = [METHOD_COLORS[m] for m in METHODS if m in METHOD_COLORS]

for row, (gname, metrics) in enumerate(plot_groups):
    for col in range(n_cols):
        ax = axes[row, col]
        if col >= len(metrics):
            ax.axis('off'); continue
        metric = metrics[col]
        vals = [df[df.method == m][metric].values for m in METHODS]
        bars = ax.bar(METHODS, [float(np.nanmean(v)) for v in vals],
                      color=palette, alpha=0.85)
        for bar, vs in zip(bars, vals):
            for v in vs:
                if not np.isnan(float(v)):
                    ax.scatter(bar.get_x() + bar.get_width()/2, float(v),
                               color='black', s=30, zorder=5)
        ax.set_title(metric)
        if col == 0: ax.set_ylabel(gname, fontsize=11)

plt.suptitle('BigNeuron Benchmark — ESA / PDS / F1 / Auxiliary', fontsize=13)
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

## MIP 오버레이 시각화

- **Gold**: 노란색 / **Ours**: 파란색 / **Vaa3D**: 주황색

In [ ]:
import tifffile
from matplotlib.collections import LineCollection

def load_swc_segments(path, scale_xy=1.0, scale_z=1.0):
    nodes = load_swc(path, scale_xy=scale_xy, scale_z=scale_z)
    segs  = []
    for nid, n in nodes.items():
        pid = n['pid']
        if pid == -1 or pid not in nodes: continue
        p = nodes[pid]
        segs.append(((n['x'],n['y'],n['z']),(p['x'],p['y'],p['z'])))
    return segs

def segs_lc_xy(segs, vox, color, lw=0.7, alpha=0.9):
    lines = [[(x1/vox,y1/vox),(x2/vox,y2/vox)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)

def segs_lc_xz(segs, vxy, vz, color, lw=0.7, alpha=0.9):
    lines = [[(x1/vxy,z1/vz),(x2/vxy,z2/vz)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)

def segs_lc_yz(segs, vxy, vz, color, lw=0.7, alpha=0.9):
    lines = [[(y1/vxy,z1/vz),(y2/vxy,z2/vz)] for (x1,y1,z1),(x2,y2,z2) in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)


def mip_figure(stem, use_original=True):
    vxy, vz = read_pixelsize(stem)

    if use_original:
        img_path = ROOT / 'data' / 'images' / f'{stem}.tif'
        swc_info = [
            ('Gold',  GOLD_DIR / f'{stem}.swc',          '#f5c518', 1.0,   1.0),
            ('Ours',  RESULTS_DIR/'ours'/f'{stem}.swc',  '#4c9be8', 1/vxy, 1/vz),
            ('Vaa3D', RESULTS_DIR/'vaa3d'/f'{stem}.swc', '#e87b4c', 1.0,   1.0),
        ]
        px_xy, px_z = 1.0, 1.0
        suffix = 'original'
    else:
        img_path = OUT_DIR / stem / 'stack_preprocessed.tif'
        meta     = np.load(str(OUT_DIR / stem / 'preprocess_meta.npz'))
        viso     = float(meta['voxel_iso'])
        swc_info = [
            ('Gold',  GOLD_DIR / f'{stem}.swc',          '#f5c518', vxy, vz),
            ('Ours',  RESULTS_DIR/'ours'/f'{stem}.swc',  '#4c9be8', 1.0, 1.0),
            ('Vaa3D', RESULTS_DIR/'vaa3d'/f'{stem}.swc', '#e87b4c', vxy, vz),
        ]
        px_xy, px_z = viso, viso
        suffix = 'preprocessed'

    if not img_path.exists():
        print(f'{stem}: {img_path.name} 없음'); return

    print(f'Loading {img_path.name} ...')
    stack  = tifffile.imread(str(img_path)).astype(np.float32)
    mip_xy = stack.max(axis=0)
    mip_xz = stack.max(axis=1)
    mip_yz = stack.max(axis=2)

    swc_segs = []
    for label, path, color, sxy, sz in swc_info:
        if not Path(path).exists():
            swc_segs.append((label, None, color)); continue
        swc_segs.append((label, load_swc_segments(path, sxy, sz), color))

    nrows = 1 + len(swc_segs)
    fig, axes = plt.subplots(nrows, 3, figsize=(15, 4*nrows))
    proj_titles = ['XY (Z-MIP)', 'XZ (Y-MIP)', 'YZ (X-MIP)']
    mips        = [mip_xy, mip_xz, mip_yz]

    for col, (mip, title) in enumerate(zip(mips, proj_titles)):
        ax   = axes[0, col]
        vmax = np.percentile(mip[mip>0], 99) if mip.max() > 0 else 1
        ax.imshow(mip, cmap='gray', origin='upper',
                  aspect='auto' if col > 0 else 'equal', vmax=vmax)
        ax.set_title(title, fontsize=11)
        if col == 0: ax.set_ylabel('Image', fontsize=11)
        ax.axis('off')

    for row, (label, segs, color) in enumerate(swc_segs, start=1):
        for col, mip in enumerate(mips):
            ax   = axes[row, col]
            vmax = np.percentile(mip[mip>0], 99) if mip.max() > 0 else 1
            ax.imshow(mip, cmap='gray', origin='upper',
                      aspect='auto' if col > 0 else 'equal', vmax=vmax)
            if segs:
                if col == 0:   ax.add_collection(segs_lc_xy(segs, px_xy, color))
                elif col == 1: ax.add_collection(segs_lc_xz(segs, px_xy, px_z, color))
                else:          ax.add_collection(segs_lc_yz(segs, px_xy, px_z, color))
            if col == 0: ax.set_ylabel(label, fontsize=11, color=color)
            ax.axis('off')

    plt.suptitle(f'{stem}  [{suffix}]  Gold(노랑)/Ours(파랑)/Vaa3D(주황)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    fname = f'mip_{stem}_{"orig" if use_original else "prep"}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

In [ ]:
for stem in samples:
    mip_figure(stem, use_original=False)
    mip_figure(stem, use_original=True)